In [ ]:
import deepinv as dinv
import torch
import matplotlib.pyplot as plt
from deepinv.utils.demo import load_dataset
from torchvision import transforms
import numpy as np
from sapg import SAPG
from priors import WaveletPrior, GSDPrior, L2, CombinedPrior, L1Prior, CRRPrior, WeightedTikhonovPrior
from prior_comparison import DegradedLikelihood
import tqdm
from utils import device
from models.utils import load_model
from scipy.stats import ortho_group
from sampling import Gaussian, SKROCK

Assume $Y = X + E$, $E\sim\mathcal{N}(0, \sigma^2I_d)$ and $X\sim\mathcal{N}(0, M)$ where $M = Q \; \mathrm{diag}(\sigma_1^2, \dots, \sigma_1^2, \sigma_2^2, \dots, \sigma_2^2) \;
 Q^T$, $Q\in O(d)$ . We have, given $X$ and $E$ are independent: $Y\sim\mathcal N(0,M')$, where $M'= Q\;\mathrm{diag}(\sigma_1^2+\sigma^2, \dots, \sigma_1^2+\sigma^2, \sigma_2^2+\sigma^2, \dots, \sigma_2^2+\sigma^2) \;
 Q^T$, i.e.
$$p(y) =(2\pi)^{-d/2} (\sigma^2+\sigma_1^2)^{-d_1}(\sigma^2+\sigma_2^2)^{-d_2/2}e^{-\frac{1}{2}y^T M'^{-1}y}$$
with $M'^{-1}=Q \; \mathrm{diag}(\frac{1}{\sigma^2+\sigma_1^2}, \dots, \frac{1}{\sigma^2+\sigma_1^2}, \frac{1}{\sigma^2+\sigma_2^2}, \dots, \frac{1}{\sigma^2+\sigma_2^2}) \;
 Q^T$ and $d=d_1 + d_2$.



We thus get
$$-\log p(y) =\frac{1}{2\sigma^2}y^T M'^{-1}y+\frac{d_1}{2}\log(\sigma_1^2+\sigma^2)+\frac{d_2}{2}\log(\sigma_2^2+\sigma^2)+\frac{d}{2}\log(2\pi).$$

$p(y^+|x)=e^{-\frac{1}{4\sigma^2}\Vert y^+-x\Vert^2}$, $x\sim p(\cdot|y-)$.

$$E=\mathbb E_\varepsilon(p(y^+/ y^-))=\mathbb E_\varepsilon(\int p(y^+/x)p(x/y^-)dx)=\int\int p(y+\varepsilon/x)\frac{p(y-\varepsilon/x)p(x)}{p(y-\varepsilon)}p(\varepsilon)dxd\varepsilon
=\int\int\frac{e^{-\frac{1}{4\sigma^2}\Vert x-y-\varepsilon\Vert^2}}{(2\pi)^{d/2}2^{d/2}\sigma^d}\frac{e^{-\frac{1}{4\sigma^2}\Vert x-y+\varepsilon\Vert^2}}{(2\pi)^{d/2}2^{d/2}\sigma^d}
\frac{e^{-\frac{1}{2}x^TM^{-1}x}}{(2\pi)^{d/2}\sqrt{\det(M)}}\frac{e^{-\frac{1}{2\sigma^2}\Vert\varepsilon\Vert^2}}{(2\pi)^{d/2}\sigma^d}
(2\pi)^{d/2}\sqrt{\det(N)}e^{\frac{1}{2}(y-\varepsilon)^TN^{-1}(y-\varepsilon)}dxd\varepsilon$$
where $N=M +2\sigma^2 I_d$, $\det N = \left(\sigma_1^2+2\sigma^2\right)^{d_1}\left(\sigma_2^2+2\sigma^2\right)^{d_2}$ and $N^{-1} = Q \; \mathrm{diag}(\frac{1}{2\sigma^2+\sigma_1^2}, \dots, \frac{1}{2\sigma^2+\sigma_1^2}, \frac{1}{2\sigma^2+\sigma_2^2}, \dots, \frac{1}{2\sigma^2+\sigma_2^2}) \;
 Q^T$, hence
 $$E = \int\int\frac{(\sigma_1^2+2\sigma^2)^{d_1/2}(\sigma_2^2+2\sigma^2)^{d_2/2}}{(2\pi)^{3d/2}2^{d}\sigma^{3d} \sigma_1^{d_1}\sigma_2^{d_2}}e^{-\frac{1}{4\sigma^2}\Vert x-y-\varepsilon\Vert^2}e^{-\frac{1}{4\sigma^2}\Vert x-y+\varepsilon\Vert^2}
e^{-\frac{1}{2}x^TM^{-1}x}e^{-\frac{1}{2\sigma^2}\Vert\varepsilon\Vert^2}
e^{\frac{1}{2}(y-\varepsilon)^TN^{-1}(y-\varepsilon)}dxd\varepsilon$$

Let $\Sigma = M^{-1}+\frac{1}{\sigma^2}I_d$ with $\det \Sigma =\left(\frac{\sigma^2+\sigma_1^2}{\sigma^2\sigma_1^2}\right)^{d_1}\left(\frac{\sigma^2+\sigma_2^2}{\sigma^2\sigma_2^2}\right)^{d_2}$ , we have
$$
\begin{array}{ll}
-\frac{1}{4\sigma^2}\Vert x-y-\varepsilon\Vert^2-\frac{1}{4\sigma^2}\Vert x-y+\varepsilon\Vert^2
-\frac{1}{2}x^TM^{-1}x&=
-\frac{1}{2\sigma^2}(\Vert x\Vert^2+\Vert y \Vert^2 +\Vert\varepsilon\Vert^2-2x\cdot y) -\frac{1}{2} x^T M^{-1} x \\
&= -\frac{1}{2}(x^T\Sigma x-\frac{2}{\sigma^2}x\cdot y) -\frac{1}{2\sigma^2}\Vert y \Vert^2 -\frac{1}{2\sigma^2}\Vert \varepsilon \Vert^2 \\
&=
-\frac{1}{2}(x-\frac{1}{\sigma^2}\Sigma^{-1}y)^T\Sigma (x-\frac{1}{\sigma^2}\Sigma^{-1}y) -\frac{1}{2\sigma^2}\Vert y \Vert^2 -\frac{1}{2\sigma^2}\Vert \varepsilon \Vert^2 + \frac{1}{2\sigma^4}y^T\Sigma^{-1}y
\end{array}
$$

thus, integrating over $x$ we get:
$$E =\frac{(\sigma_1^2+2\sigma^2)^{d_1/2}(\sigma_2^2+2\sigma^2)^{d_2/2}}{(2\pi)^{3d/2}2^{d}\sigma^{3d} \sigma_1^{d_1}\sigma_2^{d_2}}
e^{-\frac{1}{2\sigma^2}\Vert y\Vert^2+\frac{1}{2\sigma^4}y^T\Sigma^{-1}y}
\int e^{\frac{1}{2}(x-\frac{1}{\sigma^2}\Sigma^{-1}y)^T\Sigma (x-\frac{1}{\sigma^2}\Sigma^{-1}y)} dx
\int
e^{-\frac{1}{\sigma^2}\Vert\varepsilon\Vert^2}
e^{\frac{1}{2}(y-\varepsilon)^TN^{-1}(y-\varepsilon)}d\varepsilon=
\frac{(\sigma_1^2+2\sigma^2)^{d_1/2}(\sigma_2^2+2\sigma^2)^{d_2/2}}{(2\pi)^{3d/2}2^{d}\sigma^{3d} \sigma_1^{d_1}\sigma_2^{d_2}}
e^{-\frac{1}{2\sigma^2}\Vert y\Vert^2+\frac{1}{2\sigma^4}y^T\Sigma^{-1}y}
\frac{\sigma^d\sigma_1^{d_1}\sigma_2^{d_2}(2\pi)^{d/2}}{(\sigma_1^2+\sigma^2)^{d_1/2}(\sigma_2^2+\sigma^2)^{d_2/2}}
\int
e^{-\frac{1}{\sigma^2}\Vert\varepsilon\Vert^2}
e^{\frac{1}{2}(y-\varepsilon)^TN^{-1}(y-\varepsilon)}d\varepsilon$$
which reduces to:
$$E=
\frac{(\sigma_1^2+2\sigma^2)^{d_1/2}(\sigma_2^2+2\sigma^2)^{d_2/2}}{(2\pi)^{d}2^{d}\sigma^{2d}(\sigma_1^2+\sigma^2)^{d_1/2}(\sigma_2^2+\sigma^2)^{d_2/2}}
e^{-\frac{1}{2\sigma^2}\Vert y\Vert^2+\frac{1}{2\sigma^4}y^T\Sigma^{-1}y}
\int
e^{-\frac{1}{\sigma^2}\Vert\varepsilon\Vert^2+\frac{1}{2}(y-\varepsilon)^TN^{-1}(y-\varepsilon)}d\varepsilon.$$

Let $L=\frac{2}{\sigma^2}I_d-N^{-1}= Q \; \mathrm{diag}(\frac{3\sigma^2+2\sigma^2_1}{\sigma^2(2\sigma^2+\sigma_1^2)}, \dots, \frac{3\sigma^2+2\sigma^2_1}{\sigma^2(2\sigma^2+\sigma_1^2)}, \frac{3\sigma^2+2\sigma^2_2}{\sigma^2(2\sigma^2+\sigma_2^2)}, \dots, \frac{3\sigma^2+2\sigma^2_2}{\sigma^2(2\sigma^2+\sigma_2^2)}) \;
 Q^T$, we have
$$-\frac{1}{\sigma^2}\Vert\varepsilon\Vert^2+\frac{1}{2}(y-\varepsilon)^TN^{-1}(y-\varepsilon) = 
-\frac{1}{2}(\varepsilon^T L \varepsilon + y^T N^{-1} \varepsilon ) + \frac{1}{2}y^TN^{-1} = 
-\frac{1}{2}(\varepsilon + L^{-1}N^{-1}y)^T L (\varepsilon + L^{-1}N^{-1}y) + \frac{1}{2}y^TN^{-1}y+ \frac{1}{2}y^TN^{-1}L^{-1}N^{-1}y.$$
and 
$$
\begin{array}{ll}
E &= \frac{(\sigma_1^2+2\sigma^2)^{d_1/2}(\sigma_2^2+2\sigma^2)^{d_2/2}}{(2\pi)^{d}2^{d}\sigma^{2d}(\sigma_1^2+\sigma^2)^{d_1/2}(\sigma_2^2+\sigma^2)^{d_2/2}}
e^{-\frac{1}{2\sigma^2}\Vert y\Vert^2+\frac{1}{2\sigma^4}y^T\Sigma^{-1}y}
\int
e^{-\frac{1}{2}(\varepsilon + L^{-1}N^{-1}y)^T L (\varepsilon + L^{-1}N^{-1}y) + \frac{1}{2}y^TN^{-1}y+ \frac{1}{2}y^TN^{-1}L^{-1}N^{-1}y}d\varepsilon\\
&= \frac{(\sigma_1^2+2\sigma^2)^{d_1/2}(\sigma_2^2+2\sigma^2)^{d_2/2}}{(2\pi)^{d}2^{d}\sigma^{2d}(\sigma_1^2+\sigma^2)^{d_1/2}(\sigma_2^2+\sigma^2)^{d_2/2}}
e^{-\frac{1}{2\sigma^2}\Vert y\Vert^2+\frac{1}{2\sigma^4}y^T\Sigma^{-1}y+ \frac{1}{2}y^TN^{-1}y+ \frac{1}{2}y^TN^{-1}L^{-1}N^{-1}y}
\left(\frac{\sigma^2(2\sigma^2+\sigma^2_1)}{3\sigma^2+2\sigma_1^2}\right)^{d_1/2}\left(\frac{\sigma^2(2\sigma^2+\sigma^2_2)}{3\sigma^2+2\sigma_2^2}\right)^{d_2/2}.
\end{array}
$$
We finally get
$$
\begin{array}{ll}
E &= \frac{(\sigma^2+2\sigma^2_1)^{d_1}(\sigma^2+2\sigma^2_2)^{d_2}}{(2\pi)^{d/2}\sigma^d2^d\left[(\sigma^2+\sigma_1^2)(3\sigma^2+2\sigma_1^2)\right]^{d_1/2}\left[(\sigma^2+\sigma_2^2)(3\sigma^2+2\sigma_2^2)\right]^{d_2/2}}e^{-\frac{1}{2\sigma^2}\Vert y\Vert^2+\frac{1}{2\sigma^4}y^T\Sigma^{-1}y+ \frac{1}{2}y^TN^{-1}y+ \frac{1}{2}y^TN^{-1}L^{-1}N^{-1}y}\\
&=
\frac{(\sigma^2+2\sigma^2_1)^{d_1}(\sigma^2+2\sigma^2_2)^{d_2}}{(2\pi)^{d/2}\sigma^d2^d\left[(\sigma^2+\sigma_1^2)(3\sigma^2+2\sigma_1^2)\right]^{d_1/2}\left[(\sigma^2+\sigma_2^2)(3\sigma^2+2\sigma_2^2)\right]^{d_2/2}}e^{-\frac{1}{2}y^TWy}
\end{array}
$$
where $W=Q\;\mathrm{diag}(\frac{\sigma^2}{(\sigma^2+\sigma_1^2)(3\sigma^2+2\sigma_1^2)}, \dots, \frac{\sigma^2}{(\sigma^2+\sigma_1^2)(3\sigma^2+2\sigma_1^2)}, \frac{\sigma^2}{(\sigma^2+\sigma_1^2)(3\sigma^2+2\sigma_1^2)}, \dots, \frac{\sigma^2}{(\sigma^2+\sigma_1^2)(3\sigma^2+2\sigma_1^2)}) \;
 Q^T$.

In [ ]:
def create_covariance_matrix(d, v1, v2, sig, ratio=0.25):
    # create a random orthogonal matrix
    d = int(d)
    Q = ortho_group.rvs(d)
    D1  = np.zeros(d)
    ns1 = int(ratio*d)
    D1[:ns1] = v1
    D1[ns1:] = v2
    
    D_inv = np.diag(1/D1)
    N_inv = np.diag(1/(np.ones(d)/sig**2 + 1/D1)) 
    # create the covariance matrix
    return Q, D1, Q @ D_inv @ Q.T, ns1, d-ns1

In [ ]:
sigma1 = 1#1.  #  strongest deviation
sigma2 = 0.1#0.1
v1, v2 = sigma1**2, sigma2**2  
noise_level = 0.05#0.05

def generate_measurements(d, v1, v2):
    d = int(d)
    Q, D, cov_inv, d1, d2 = create_covariance_matrix(d, v1, v2, noise_level)
    x = torch.tensor((Q @ np.sqrt(np.diag(D)) @ np.random.normal(size=d)).astype(np.float32)).to(device).reshape((1, 1, d))
    
    p = dinv.physics.LinearPhysics(  # identity, for compatibility
        img_size=(1, d),
        device=device)
    
    y = p(x) + torch.tensor(noise_level*np.random.normal(size=d).astype(np.float32)).to(device)
    return (y, x, p, d1, d2,
            torch.tensor(cov_inv.astype(np.float32), device=device),
            Q, D)

def compute_evidence(d1, d2, v1, v2, noise_level, y, D, Q, mlog=True):  # sum of X and a gaussian of variance noise_level^2
    d = d1 + d2
    # compute -log p
    cov_inv = Q @ np.diag(1 / (D + noise_level**2))  @ Q.T
    res = 0.5*np.sum(np.reshape(y.cpu().numpy(), (-1, 1))*(cov_inv@np.reshape(y.cpu().numpy(), (-1, 1))))
    res += d1*0.5*np.log(v1 + noise_level**2)  # first part of 1/sqrt(det N)
    res += d2*0.5*np.log(v2 + noise_level**2)  # second part of sqrt(det N)
    res += 0.5*d*np.log(2*np.pi)  # normalization of likelihood term

    if mlog == False:
        res = np.exp(-res)
    return res


def compute_evidence2(d1, d2, v1, v2, noise_level, y, D, Q, mlog=True):
    d = d1 + d2
    mat_comb = Q @ np.diag(noise_level**2/((noise_level**2+D)*(3*noise_level**2 + 2*D))) @ Q.T
    
    res = 0.5*np.sum(np.reshape(y.cpu().numpy(), (-1, 1))*(mat_comb@np.reshape(y.cpu().numpy(), (-1, 1))))
    res -= d1*np.log(noise_level**2 + 2*v1) + d2*np.log(noise_level**2 + 2*v2)     
    res += 0.5*d*np.log(2*np.pi) + d*np.log(2) + d*np.log(noise_level)
    res += 0.5*d1*np.log((noise_level**2+v1)*(3*noise_level**2 + 2*v1)) + 0.5*d2*np.log((noise_level**2+v2)*(3*noise_level**2 + 2*v2))

    if mlog == False:
        return np.exp(-res)
    else:
        return res
        
# Lipschitz constant of nabla f
L_f = 1 / noise_level**2 / 2.  # divide by 2 because std is sqrt(2)sigma

In [ ]:
# Lipschitz constant of grad g, which is reg_str * norm(cov-1) = reg_str * norm(cov-1)
L_g = max(1/v1, 1/v2)
L =  L_f + L_g

# Stepsize of MCMC algorithm
gamma = 0.98*1/L
print("gamma: {}".format(gamma))

### Loop over d

In [ ]:
torch.manual_seed(0)
np.random.seed(0)

nval = 5
dims = torch.linspace(5, 1000., nval, dtype=torch.int)#torch.logspace(2, 4., nval, dtype=torch.int)
hist, evidences = np.zeros(nval), np.zeros(nval)
nb_steps, burnin_ratio = 40000, 0.98#600000, 0.9
post_hist_traces = []
evidences2 = np.zeros(nval)

for i in range(nval):
    d = dims[i]
    y, x, p, d1, d2, cov_inv, Q, D = generate_measurements(d, v1, v2)
    g = WeightedTikhonovPrior(torch.tensor(1.), cov_inv)
    L_g = d * max(1/v1, 1/v2)
    gamma = 0.98*1/(L_f + L_g)
    D2 = np.ones(d)*np.sqrt(2)*noise_level
    #dl = DegradedLikelihood(y, g, p, noise_level, gamma, X_init=p.A_A_adjoint(y).to(device).clone(),
     #                       lam_reg=None, project=None, sampler=Gaussian, sampler_kwargs=dict(D=torch.tensor(D2.astype(np.float32), device=device), Q=torch.tensor(Q.astype(np.float32), device=device)))
    dl = DegradedLikelihood(y, g, p, noise_level, gamma, X_init=p.A_A_adjoint(y).to(device).clone(),sampler=SKROCK,sampler_kwargs={'s':15},
                            lam_reg=None, project=None)
    print("d={}".format(d))
    
    X_post_trace, post_hist, lik_trace, lik_mean = dl.compute_test2(nb_steps, burnin_ratio=0.9, log_stats=True, thinning=100, tol=1e-4, patience=20)#    X_post_trace, post_hist, lik_trace, lik_mean = dl.compute_test2(nb_steps, burnin_ratio=0.9, log_stats=True, thinning=50, tol=1e-2, patience=10)

    evidences[i] = compute_evidence(d1, d2, v1, v2, noise_level, y, D, Q, mlog=True)
    print(-np.log(lik_mean.cpu())+d*(np.log(noise_level*np.sqrt(2)) + 0.5*np.log(2*np.pi)))
    hist[i] = -np.log(lik_mean.cpu())+d*(np.log(noise_level*np.sqrt(2)) + 0.5*np.log(2*np.pi))#lik_mean #+ 0.5*d*np.log(2*np.pi) + d*np.log(np.sqrt(2)*noise_level)  # lik_mean is E_eps(p(y+/y-)) without normalization constant
    evidences2[i] = compute_evidence2(d1, d2, v1, v2, noise_level, y, D, Q, mlog=True)

    #post_hist_traces.append(post_hist.numpy())
    print("likelihood: ", hist[i], "  evidence: ", evidences[i], evidences2[i])

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(25,5))

ax[0].plot(dims, -hist, label=r'$\log\mathbb{E}_\varepsilon(p(y^+/\;y^-))$ MC')
ax[0].plot(dims, -evidences, '--', label=r'$\log p(y)$')
ax[0].plot(dims, -evidences2, '--', label=r'$\log \mathbb{E}_\varepsilon(p(y^+/\;y^-))$')

ax[0].set_xlabel(r'$d$')
c1, c2 = np.array((0.1, 0.2, 0.5)), np.array((0.9, 0.3, 0.55))
for i in range(len(post_hist_traces)):
    if i == 0 or i == nval - 1:
        label = r'$d = {:.3f}'.format(dims[i])
    else:
        label = None
    t = i / nval
    ax[1].plot(post_hist_traces[i], color=(t*c1 + (1-t)*c2), linestyle='-.',label=label)

ax[0].legend(), ax[1].legend()
ax[0].set_yscale('log')

plt.savefig('figs/test2_evidence7.pdf')